In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

drive_folder = "/content/drive/MyDrive"

print("Available dataset ZIP files:\n")

for file in os.listdir(drive_folder):
    if file.lower().endswith(".zip"):
        print(file)

In [ ]:
import zipfile

zip_files = [
    "/content/drive/MyDrive/Lung X-Ray.zip",
    "/content/drive/MyDrive/Chest X-Ray Images for 5-Class Lung Disease Classi.zip"
]

for zip_path in zip_files:
    print("\nDataset:", zip_path.split("/")[-1])

    with zipfile.ZipFile(zip_path, "r") as z:
        for name in z.namelist():
            if name.lower().endswith(
                (".csv", ".txt", ".json", ".xlsx", ".pdf")
            ):
                print("Documentation:", name)

In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/Chest X-Ray Images for 5-Class Lung Disease Classi.zip"
extract_path = "/content/lung_pneumonia_emphysema"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

In [ ]:
import os

for root, dirs, files in os.walk("/content/lung_pneumonia_emphysema"):
    level = root.replace("/content/lung_pneumonia_emphysema", "").count(os.sep)

    if level <= 3:
        print("  " * level + os.path.basename(root) + "/")

        if level == 3:
            print("  " * (level + 1) + "Images:", len(files))

In [ ]:
import os

base_path = "/content/lung_pneumonia_emphysema/Chest X-Ray Images for 5-Class Lung Disease Classi"

for class_name in ["Pneumonia", "Emphysema"]:
    folder = os.path.join(base_path, class_name)
    images = [
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]
    print(class_name, ":", len(images), "images")

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("Libraries imported successfully!")
print("TensorFlow version:", tf.__version__)

In [ ]:
import glob

pneumonia_path = os.path.join(base_path, "Pneumonia")
emphysema_path = os.path.join(base_path, "Emphysema")

pneumonia_images = glob.glob(pneumonia_path + "/*")
emphysema_images = glob.glob(emphysema_path + "/*")

image_paths = pneumonia_images + emphysema_images
labels = [1] * len(pneumonia_images) + [0] * len(emphysema_images)

print("Total images:", len(image_paths))
print("Pneumonia images:", len(pneumonia_images))
print("Emphysema images:", len(emphysema_images))

In [ ]:
# First, separate 70% training and 30% temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    image_paths,
    labels,
    test_size=0.30,
    random_state=42,
    stratify=labels
)

# Split the remaining 30% equally
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training images:", len(X_train))
print("Validation images:", len(X_val))
print("Testing images:", len(X_test))

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(
        image, channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))

    return image, tf.cast(label, tf.float32)

print("Image loading function created successfully!")

In [ ]:
def make_dataset(paths, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        dataset = dataset.shuffle(buffer_size=len(paths))

    dataset = dataset.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)

print("Training, validation, and testing datasets created successfully!")

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

print("EfficientNetB0 loaded successfully!")

In [ ]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid")
])

print("Classification model created successfully!")

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

In [ ]:
import json

history_path = "/content/drive/MyDrive/Lung_Project_Dataset/training_history.json"

with open(history_path, "w") as f:
    json.dump(history.history, f)

print("Training history saved successfully!")

In [ ]:
model_path = "/content/drive/MyDrive/Lung_Project_Dataset/lung_model.keras"

model.save(model_path)

print("Trained model saved successfully!")

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy * 100, "%")

In [ ]:
y_pred_prob = model.predict(test_ds)

y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print("Predictions generated successfully!")
print("Total predictions:", len(y_pred))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = np.array(y_test)

print("CLASSIFICATION REPORT")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Emphysema", "Pneumonia"]
))

cm = confusion_matrix(y_true, y_pred)

print("CONFUSION MATRIX")
print(cm)

In [ ]:
import json

results = {
    "test_accuracy": float(test_accuracy),
    "test_loss": float(test_loss),
    "classification_report": classification_report(
        y_true,
        y_pred,
        target_names=["Emphysema", "Pneumonia"],
        output_dict=True
    ),
    "confusion_matrix": cm.tolist()
}

results_path = "/content/drive/MyDrive/Lung_Project_Dataset/evaluation_results.json"

with open(results_path, "w") as f:
    json.dump(results, f, indent=4)

print("Evaluation results saved successfully!")

In [ ]:
for layer in base_model.layers[-15:]:
    print(layer.name)

In [ ]:
print("Model input shape:", model.inputs[0].shape)
print("Model prediction:", model.predict(
    np.zeros((1, 224, 224, 3), dtype=np.float32)
))

In [ ]:
grad_model = tf.keras.Model(
    inputs=base_model.input,
    outputs=base_model.get_layer("top_conv").output
)

print("Grad-CAM feature model created successfully!")

In [ ]:
def make_gradcam_heatmap(img_array):
    with tf.GradientTape() as tape:
        features = grad_model(img_array, training=False)
        tape.watch(features)

        predictions = model(img_array, training=False)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, features)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    heatmap = tf.reduce_sum(
        features[0] * pooled_grads,
        axis=-1
    )

    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()

print("Grad-CAM heatmap function created successfully!")

In [ ]:
# Select one Pneumonia image
sample_image_path = X_test[list(y_test).index(1)]

# Load and resize the image
img = tf.keras.utils.load_img(
    sample_image_path,
    target_size=(224, 224)
)

# Convert image to array
img_array = tf.keras.utils.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

print("Sample X-ray image loaded successfully!")
print("Image shape:", img_array.shape)

In [ ]:
def make_gradcam_heatmap(img_array):
    with tf.GradientTape() as tape:
        features = grad_model(img_array, training=False)
        tape.watch(features)

        # Pass the extracted features through the classifier layers
        x = model.layers[1](features)
        x = model.layers[2](x)
        x = model.layers[3](x)
        x = model.layers[4](x)
        predictions = model.layers[5](x, training=False)

        loss = predictions[:, 0]

    grads = tape.gradient(loss, features)

    if grads is None:
        raise ValueError("Gradients are not connected. Check model layers.")

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    heatmap = tf.reduce_sum(features[0] * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap) + 1e-8

    return heatmap.numpy()

print("Grad-CAM function updated successfully!")

In [ ]:
heatmap = make_gradcam_heatmap(img_array)

print("Grad-CAM heatmap generated successfully!")
print("Heatmap shape:", heatmap.shape)

In [ ]:
import cv2

# Resize heatmap to image dimensions
heatmap_resized = cv2.resize(heatmap, (224, 224))

# Convert heatmap into a color map
heatmap_colored = cv2.applyColorMap(
    np.uint8(255 * heatmap_resized),
    cv2.COLORMAP_JET
)

# Convert original image to uint8
original_img = np.uint8(img_array[0])

# Overlay heatmap on original image
superimposed_img = cv2.addWeighted(
    original_img, 0.6,
    heatmap_colored, 0.4,
    0
)

# Display results
plt.figure(figsize=(12, 5))

plt.subplot(1, 3, 1)
plt.imshow(original_img)
plt.title("Original X-ray")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(heatmap_resized, cmap="jet")
plt.title("Grad-CAM Heatmap")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
plt.title("Grad-CAM Overlay")
plt.axis("off")

plt.show()

In [ ]:
prediction = model.predict(img_array)[0][0]

predicted_class = "Pneumonia" if prediction >= 0.5 else "Emphysema"

print("Predicted Class:", predicted_class)
print("Prediction Score:", prediction)

In [ ]:
gradcam_path = "/content/drive/MyDrive/Lung_Project_Dataset/gradcam_result.png"

plt.figure(figsize=(10, 5))
plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
plt.title("Grad-CAM - Pneumonia")
plt.axis("off")
plt.savefig(gradcam_path, bbox_inches="tight")
plt.show()

print("Grad-CAM image saved successfully!")

In [ ]:
prediction_result = {
    "predicted_class": predicted_class,
    "prediction_score": float(prediction),
    "image_path": sample_image_path,
    "gradcam_image": gradcam_path
}

prediction_path = "/content/drive/MyDrive/Lung_Project_Dataset/prediction_result.json"

with open(prediction_path, "w") as f:
    json.dump(prediction_result, f, indent=4)

print("Prediction details saved successfully!")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
graph_path = "/content/drive/MyDrive/Lung_Project_Dataset/training_graphs.png"

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.savefig(graph_path, dpi=300, bbox_inches="tight")
plt.show()

print("Training graphs saved successfully!")

In [ ]:
import os

folder = "/content/drive/MyDrive/Lung_Project_Dataset"

for file in os.listdir(folder):
    file_path = os.path.join(folder, file)
    print(file, "-", round(os.path.getsize(file_path) / (1024 * 1024), 2), "MB")

In [ ]:
loaded_model = tf.keras.models.load_model(
    "/content/drive/MyDrive/Lung_Project_Dataset/lung_model.keras"
)

print("Saved model loaded successfully!")

loaded_model.summary()

In [ ]:
loaded_prediction = loaded_model.predict(img_array)[0][0]

loaded_class = (
    "Pneumonia" if loaded_prediction >= 0.5
    else "Emphysema"
)

print("Loaded Model Prediction:", loaded_class)
print("Prediction Score:", loaded_prediction)

In [ ]:
import hashlib

def get_image_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

train_hashes = set(get_image_hash(p) for p in X_train)
test_hashes = set(get_image_hash(p) for p in X_test)

duplicates = train_hashes.intersection(test_hashes)

print("Duplicate images across training and testing:", len(duplicates))

In [ ]:
duplicate_hash = list(duplicates)[0]

train_duplicate = next(
    p for p in X_train
    if get_image_hash(p) == duplicate_hash
)

test_duplicate = next(
    p for p in X_test
    if get_image_hash(p) == duplicate_hash
)

print("Training duplicate:", train_duplicate)
print("Testing duplicate:", test_duplicate)

In [ ]:
keep_indices = [
    i for i, path in enumerate(X_test)
    if get_image_hash(path) != duplicate_hash
]

X_test_clean = [X_test[i] for i in keep_indices]
y_test_clean = [y_test[i] for i in keep_indices]

print("Original test images:", len(X_test))
print("Clean test images:", len(X_test_clean))
print("Removed duplicates:", len(X_test) - len(X_test_clean))

In [ ]:
clean_test_ds = make_dataset(X_test_clean, y_test_clean)

clean_loss, clean_accuracy = loaded_model.evaluate(clean_test_ds)

print("Clean Test Loss:", clean_loss)
print("Clean Test Accuracy:", clean_accuracy * 100, "%")

In [ ]:
clean_results = {
    "test_images": len(X_test_clean),
    "test_loss": float(clean_loss),
    "test_accuracy": float(clean_accuracy),
    "duplicates_removed": 1
}

clean_results_path = "/content/drive/MyDrive/Lung_Project_Dataset/clean_evaluation_results.json"

with open(clean_results_path, "w") as f:
    json.dump(clean_results, f, indent=4)

print("Clean evaluation results saved successfully!")

In [ ]:
import os

folder = "/content/drive/MyDrive/Lung_Project_Dataset"

print("FINAL PROJECT FILES\n")

for file in sorted(os.listdir(folder)):
    print("✓", file)

In [ ]:
requirements = """tensorflow
numpy
matplotlib
scikit-learn
pillow
"""

with open("/content/requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created!")

In [ ]:
from google.colab import files
files.download("/content/requirements.txt")